# Chapitre 1 — Python scientifique et NumPy

**Cours : Analyse et traitement des données** · Auteur : *Issa Gueye*

## Objectifs

- Comprendre pourquoi **NumPy** est la fondation de tout l'écosystème (pandas, scikit-learn, SciPy…).
- Créer, indexer et transformer des **tableaux** (`ndarray`).
- Maîtriser la **vectorisation** et le **broadcasting** pour écrire du code rapide et lisible.
- Calculer des statistiques descriptives et générer des nombres aléatoires **reproductibles**.

In [1]:
import time

import numpy as np

np.set_printoptions(precision=3, suppress=True)

## 1. Pourquoi NumPy ?

Une liste Python contient des **références vers des objets** dispersés en mémoire.
Un `ndarray` NumPy contient des **valeurs de même type stockées de façon contiguë** :
les opérations sont exécutées en C, sans boucle Python.

In [2]:
n = 1_000_000
liste = list(range(n))
tableau = np.arange(n)

t0 = time.perf_counter()
carres_liste = [x * x for x in liste]
t1 = time.perf_counter()
carres_np = tableau * tableau
t2 = time.perf_counter()

print(f"Boucle Python : {(t1 - t0) * 1000:7.1f} ms")
print(f"NumPy         : {(t2 - t1) * 1000:7.1f} ms  (≈ {(t1 - t0) / (t2 - t1):.0f}× plus rapide)")

Boucle Python :    43.5 ms
NumPy         :     3.5 ms  (≈ 12× plus rapide)


## 2. Créer des tableaux

In [3]:
a = np.array([12500, 6500, 700, 2300])          # depuis une liste
z = np.zeros((2, 3))                             # matrice de zéros
u = np.ones(4, dtype=np.int32)                   # type explicite
r = np.arange(0, 1, 0.25)                        # comme range, avec un pas réel
l = np.linspace(0, 1, 5)                         # 5 points régulièrement espacés, bornes incluses
I = np.eye(3)                                    # matrice identité
print(a, a.dtype, a.shape, a.ndim, sep="\n")
print(l)

[12500  6500   700  2300]
int64
(4,)
1
[0.   0.25 0.5  0.75 1.  ]


| Attribut | Signification |
|---|---|
| `shape` | dimensions, ex. `(lignes, colonnes)` |
| `ndim` | nombre d'axes |
| `dtype` | type des éléments (`int64`, `float64`, `bool`, …) |
| `size` | nombre total d'éléments |

⚠️ Un tableau n'a **qu'un seul dtype**. Mélanger entiers et texte convertit tout en texte :

In [4]:
print(np.array([1, 2, "3"]).dtype)   # <U21 : chaînes Unicode !

<U21


## 3. Indexation et découpage

In [5]:
M = np.arange(1, 13).reshape(3, 4)
print(M)
print("Élément (1, 2)          :", M[1, 2])
print("2e ligne                :", M[1])
print("3e colonne              :", M[:, 2])
print("Sous-matrice            :\n", M[:2, 1:3])
print("Une ligne sur deux      :\n", M[::2])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Élément (1, 2)          : 7
2e ligne                : [5 6 7 8]
3e colonne              : [ 3  7 11]
Sous-matrice            :
 [[2 3]
 [6 7]]
Une ligne sur deux      :
 [[ 1  2  3  4]
 [ 9 10 11 12]]


### Indexation booléenne (filtrage) — l'outil le plus utilisé en analyse

In [6]:
temperatures = np.array([24.1, 27.3, -99.0, 29.8, 31.2, np.nan, 26.4])
valides = (temperatures > -50) & ~np.isnan(temperatures)   # & = ET, | = OU, ~ = NON
print(valides)
print("Températures valides :", temperatures[valides])
print("Moyenne valide        :", temperatures[valides].mean())

[ True  True False  True  True False  True]
Températures valides : [24.1 27.3 29.8 31.2 26.4]
Moyenne valide        : 27.76


> ⚠️ Utilisez `&`, `|`, `~` (et des parenthèses), **pas** `and`, `or`, `not`, sur des tableaux.

### Vue ou copie ?
Un découpage (`M[:2]`) renvoie une **vue** : modifier la vue modifie l'original.
L'indexation booléenne ou par liste d'indices renvoie une **copie**.

In [7]:
v = M[0]          # vue
v[0] = 100
print(M[0])       # l'original a changé !
M[0, 0] = 1       # on remet
c = M[[0]].copy() # copie explicite : sûre

[100   2   3   4]


## 4. Vectorisation et fonctions universelles (ufuncs)

Les opérations s'appliquent **élément par élément** sans boucle :

In [8]:
prix = np.array([12500, 6500, 700, 2300])
quantites = np.array([3, 2, 10, 4])
chiffre_affaires = prix * quantites
print(chiffre_affaires, chiffre_affaires.sum())
print(np.log(prix).round(2))
print(np.sqrt(quantites))
print(np.where(chiffre_affaires > 10000, "gros", "petit"))   # « si … alors … sinon » vectorisé

[37500 13000  7000  9200] 66700
[9.43 8.78 6.55 7.74]
[1.732 1.414 3.162 2.   ]
['gros' 'gros' 'petit' 'petit']


## 5. Broadcasting

Quand deux tableaux n'ont pas la même forme, NumPy **étire virtuellement** les dimensions de taille 1.

**Règle** : on compare les formes **de droite à gauche** ; deux dimensions sont compatibles si elles
sont **égales** ou si l'une vaut **1**.

```text
  (3, 4)   tableau de données : 3 boutiques × 4 produits
     (4,)  prix des 4 produits      → étiré en (3, 4)
  ------
  (3, 4)   résultat
```

In [9]:
ventes_qte = np.array([[3, 2, 10, 4],
                       [1, 0, 25, 2],
                       [5, 3, 7, 0]])           # 3 boutiques × 4 produits
ca = ventes_qte * prix                          # (3,4) * (4,) -> (3,4)
print(ca)
print("CA par boutique :", ca.sum(axis=1))      # axis=1 : on somme le long des colonnes
print("CA par produit  :", ca.sum(axis=0))      # axis=0 : on somme le long des lignes

[[37500 13000  7000  9200]
 [12500     0 17500  4600]
 [62500 19500  4900     0]]
CA par boutique : [66700 34600 86900]
CA par produit  : [112500  32500  29400  13800]


### Application : standardiser chaque colonne (centrer-réduire)

$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$

In [10]:
X = np.random.default_rng(0).normal(loc=[50, 10_000, 3], scale=[10, 2_000, 1], size=(200, 3))
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)   # (200,3) - (3,) : broadcasting
print("moyennes ≈ 0 :", Z.mean(axis=0).round(10))
print("écarts-types = 1 :", Z.std(axis=0, ddof=1))

moyennes ≈ 0 : [-0.  0.  0.]
écarts-types = 1 : [1. 1. 1.]


> 📝 `ddof=1` donne l'écart-type **corrigé** (division par $n-1$), estimateur sans biais de la variance.
> Par défaut NumPy utilise `ddof=0`, pandas utilise `ddof=1` : source fréquente d'écarts entre les deux !

## 6. Statistiques descriptives et valeurs manquantes

In [11]:
x = np.array([3.0, 7.0, np.nan, 1.0, 9.0])
print(np.mean(x))                       # nan : NaN « contamine » le calcul
print(np.nanmean(x), np.nanmedian(x), np.nanstd(x, ddof=1))
print(np.nanpercentile(x, [25, 50, 75]))

nan
5.0 5.0 3.6514837167011076
[2.5 5.  7.5]


## 7. Aléatoire reproductible

Utilisez **toujours** un générateur explicite avec une graine (`seed`) :
les résultats sont alors **reproductibles** par n'importe qui.

In [12]:
rng = np.random.default_rng(seed=42)
print(rng.normal(0, 1, 3))
print(rng.integers(1, 7, size=10))                   # 10 lancers de dé
print(rng.choice(["Wave", "Orange Money", "Espèces"], size=5, p=[0.3, 0.2, 0.5]))
echantillon = rng.permutation(10)                      # mélange
print(echantillon)

[ 0.305 -1.04   0.75 ]
[1 5 2 1 4 6 5 5 5 5]
['Wave' 'Orange Money' 'Orange Money' 'Espèces' 'Espèces']
[1 0 5 4 8 9 7 2 3 6]


### Illustration : la loi des grands nombres
La moyenne de lancers de dé converge vers 3,5.

In [13]:
lancers = rng.integers(1, 7, size=100_000)
moyennes_cumulees = np.cumsum(lancers) / np.arange(1, lancers.size + 1)
for k in [10, 100, 1_000, 10_000, 100_000]:
    print(f"n = {k:>7} : moyenne = {moyennes_cumulees[k - 1]:.4f}")

n =      10 : moyenne = 3.6000
n =     100 : moyenne = 3.5300
n =    1000 : moyenne = 3.4760
n =   10000 : moyenne = 3.4747
n =  100000 : moyenne = 3.4997


## 8. Un peu d'algèbre linéaire

Utile pour la régression (chapitre 9) et l'ACP.
Résolvons les **moindres carrés** $\hat\beta = \arg\min \|y - X\beta\|^2$, dont la solution vérifie
$X^\top X \hat\beta = X^\top y$.

In [14]:
rng = np.random.default_rng(1)
heures = rng.uniform(0, 20, 100)
note = 6 + 0.4 * heures + rng.normal(0, 1.5, 100)
X = np.column_stack([np.ones_like(heures), heures])   # colonne de 1 pour l'ordonnée à l'origine
beta, *_ = np.linalg.lstsq(X, note, rcond=None)      # plus stable que inv(X.T @ X) @ X.T @ y
print(f"note ≈ {beta[0]:.2f} + {beta[1]:.3f} × heures")

note ≈ 5.90 + 0.401 × heures


## À retenir

| Besoin | Outil |
|---|---|
| Calcul rapide sur des nombres | `ndarray` + opérations vectorisées |
| Filtrer | masque booléen `x[(x > a) & (x < b)]` |
| Opérations entre formes différentes | broadcasting (comparer les formes de droite à gauche) |
| Résumer par ligne / colonne | `axis=1` / `axis=0` |
| Ignorer les NaN | `np.nanmean`, `np.nanmedian`, … |
| Aléatoire reproductible | `rng = np.random.default_rng(seed)` |

**Références** : Harris C. R. et al. (2020), « Array programming with NumPy », *Nature* 585 ;
documentation officielle <https://numpy.org/doc/stable/>.

➡️ **Chapitre suivant : pandas, les fondamentaux.**